# Final Fusion Meta Model Ablation

Ce notebook reprend le script `final_fusion_meta_model_ablation.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Compare des familles de fusion finale avant de les tester en mode causal.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Meta-model ablation for final trajectory+attention+PPE fusion scores.
- Commande de reproduction referencee : final fusion meta-model ablation.
- Artefacts controles : Final fusion meta-model family ablation exists. (`runs/exp_068_final_fusion_meta_model_ablation/metrics/fusion_meta_model_metric_summary.csv`).
- Run par defaut : `runs/exp_068_final_fusion_meta_model_ablation`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "final_fusion_meta_model_ablation.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_pipeline import ROOT, safe_auc, threshold_sweep, write_json
from sequence_experiments import append_report, make_run_dir


try:
    from xgboost import XGBClassifier
except Exception:  # pragma: no cover - optional dependency
    XGBClassifier = None


BASELINE_SCORE_COLS = [
    "final_sequence_only",
    "final_attention_rule",
    "final_ppe_rule",
    "final_full_rule",
    "final_safety_sensitive_rule",
    "final_high_sensitivity_rule",
    "final_attention_ppe_prior",
    "final_learned_meta_mean",
]


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    return path if path.is_absolute() else ROOT / path


## Fonction `ece_score`

Cette cellule definit `ece_score`. Elle prepare une partie du script.

In [ ]:
def ece_score(y, p, bins=10):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    total = max(1, len(y))
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        if hi == 1.0:
            mask = (p >= lo) & (p <= hi)
        else:
            mask = (p >= lo) & (p < hi)
        if not mask.any():
            continue
        ece += float(mask.sum()) / total * abs(float(y[mask].mean()) - float(p[mask].mean()))
    return float(ece)


## Fonction `feature_frame`

Cette cellule definit `feature_frame`. Elle prepare une partie du script.

In [ ]:
def feature_frame(df, feature_set):
    base = pd.DataFrame(index=df.index)
    d = df["final_sequence_only"].clip(0, 1)
    a = df["attention_risk"].clip(0, 1)
    p = df["ppe_risk"].clip(0, 1)
    base["sequence_mean"] = d
    base["sequence_max"] = df.get("sequence_max_tcn", d).clip(0, 1)
    base["attention_risk"] = a
    base["ppe_risk"] = p
    base["d_x_attention"] = d * a
    base["d_x_ppe"] = d * p
    base["attention_x_ppe"] = a * p
    base["max_attention_ppe"] = np.maximum(a, p)
    base["attention_ppe_prior"] = df["final_attention_ppe_prior"].clip(0, 1)
    if feature_set == "core":
        return base
    if feature_set == "all_scores":
        extra_cols = [
            col
            for col in df.columns
            if col.startswith("sequence_")
            or col.startswith("learned_meta_")
            or col in ["final_attention_rule", "final_ppe_rule", "final_full_rule", "final_safety_sensitive_rule"]
        ]
        extra = df[extra_cols].copy()
        return pd.concat([base, extra], axis=1)
    raise ValueError(feature_set)


## Fonction `positive_proba`

Cette cellule definit `positive_proba`. Elle prepare une partie du script.

In [ ]:
def positive_proba(model, X):
    proba = model.predict_proba(X)
    classes = getattr(model, "classes_", None)
    if classes is None and hasattr(model, "named_steps"):
        final = model.named_steps[list(model.named_steps.keys())[-1]]
        classes = getattr(final, "classes_", None)
    if classes is None:
        return proba[:, 1]
    return proba[:, list(classes).index(1)]


## Fonction `class_weights`

Cette cellule definit `class_weights`. Elle prepare une partie du script.

In [ ]:
def class_weights(y):
    positives = max(1, int(np.sum(y == 1)))
    negatives = max(1, int(np.sum(y == 0)))
    weights = np.ones(len(y), dtype=float)
    weights[y == 1] = negatives / positives
    return weights


## Fonction `build_model`

Cette cellule definit `build_model`. Elle prepare une partie du script.

In [ ]:
def build_model(model_name, seed, y_train):
    if model_name == "logistic_l2":
        return make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=3000, C=0.5, random_state=seed))
    if model_name == "logistic_elasticnet":
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(
                class_weight="balanced",
                penalty="elasticnet",
                solver="saga",
                l1_ratio=0.35,
                max_iter=5000,
                C=0.4,
                random_state=seed,
            ),
        )
    if model_name == "gaussian_nb":
        return GaussianNB()
    if model_name == "random_forest":
        return RandomForestClassifier(
            n_estimators=400,
            max_depth=4,
            min_samples_leaf=8,
            class_weight="balanced_subsample",
            random_state=seed,
            n_jobs=-1,
        )
    if model_name == "extra_trees":
        return ExtraTreesClassifier(
            n_estimators=500,
            max_depth=5,
            min_samples_leaf=6,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1,
        )
    if model_name == "hist_gbdt":
        return HistGradientBoostingClassifier(
            max_iter=150,
            learning_rate=0.035,
            max_leaf_nodes=8,
            l2_regularization=0.2,
            random_state=seed,
        )
    if model_name == "mlp_small":
        return make_pipeline(
            StandardScaler(),
            MLPClassifier(
                hidden_layer_sizes=(16,),
                alpha=0.02,
                learning_rate_init=0.002,
                max_iter=700,
                early_stopping=True,
                validation_fraction=0.20,
                random_state=seed,
            ),
        )
    if model_name == "xgb_conservative":
        if XGBClassifier is None:
            return None
        positives = max(1, int(np.sum(y_train == 1)))
        negatives = max(1, int(np.sum(y_train == 0)))
        return XGBClassifier(
            n_estimators=180,
            max_depth=2,
            learning_rate=0.035,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.2,
            reg_lambda=3.0,
            min_child_weight=4.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            scale_pos_weight=negatives / positives,
            random_state=seed,
            n_jobs=2,
        )
    raise ValueError(model_name)


## Fonction `fit_predict_meta`

Cette cellule definit `fit_predict_meta`. Elle prepare une partie du script.

In [ ]:
def fit_predict_meta(df, seed, feature_set, model_name, run_dir):
    y = df["danger_within_1.0s"].astype(int).to_numpy()
    train_mask = df["split"].eq("train").to_numpy()
    X = feature_frame(df, feature_set)
    X_train = X.loc[train_mask]
    y_train = y[train_mask]

    if model_name == "isotonic_sequence":
        model = IsotonicRegression(out_of_bounds="clip")
        model.fit(df.loc[train_mask, "final_sequence_only"].to_numpy(), y_train)
        probs = model.predict(df["final_sequence_only"].to_numpy())
    else:
        model = build_model(model_name, seed, y_train)
        if model is None:
            return None
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            if model_name in {"hist_gbdt", "xgb_conservative", "gaussian_nb"}:
                try:
                    model.fit(X_train, y_train, sample_weight=class_weights(y_train))
                except TypeError:
                    model.fit(X_train, y_train)
            else:
                model.fit(X_train, y_train)
        probs = positive_proba(model, X)

    probs = np.clip(np.asarray(probs, dtype=float), 0.0, 1.0)
    model_dir = run_dir / "models"
    model_dir.mkdir(exist_ok=True, parents=True)
    joblib.dump(
        {
            "model": model,
            "feature_set": feature_set,
            "model_name": model_name,
            "feature_columns": ["final_sequence_only"] if model_name == "isotonic_sequence" else list(X.columns),
        },
        model_dir / f"fusion_meta_seed{seed}_{feature_set}_{model_name}.joblib",
    )
    return probs


## Fonction `evaluate_score`

Cette cellule definit `evaluate_score`. Elle prepare une partie du script.

In [ ]:
def evaluate_score(df, score_col, variant, seed, persistence_windows):
    rows = []
    sweeps = []
    for split in ["train", "val", "test"]:
        split_df = df[df["split"].eq(split)].copy()
        y = split_df["danger_within_1.0s"].astype(int).to_numpy()
        p = split_df[score_col].astype(float).to_numpy()
        sweep = threshold_sweep(split_df.rename(columns={score_col: "risk"}), "risk", 1.0, split, persistence_windows=persistence_windows)
        sweep["repeat_seed"] = seed
        sweep["variant"] = variant
        sweeps.append(sweep)
        best = sweep.sort_values(["danger_clip_hit_rate", "safe_false_alarms_per_min", "window_precision"], ascending=[False, True, False]).iloc[0]
        rows.append(
            {
                "repeat_seed": seed,
                "variant": variant,
                "split": split,
                "n": int(len(split_df)),
                "positives": int(y.sum()),
                "average_precision": safe_auc(average_precision_score, y, p),
                "roc_auc": safe_auc(roc_auc_score, y, p),
                "brier": float(brier_score_loss(y, p)) if len(np.unique(y)) > 1 else np.nan,
                "ece_10bin": ece_score(y, p),
                "best_threshold_by_hit_fa": float(best["threshold"]),
                "best_hit_rate": float(best["danger_clip_hit_rate"]),
                "best_false_alarms_per_min": float(best["safe_false_alarms_per_min"]),
                "best_window_precision": float(best["window_precision"]),
                "best_window_f1": float(best["window_f1"]),
                "best_median_early_warning_s": best["median_early_warning_s"],
            }
        )
    return rows, sweeps


## Fonction `select_validation_threshold`

Cette cellule definit `select_validation_threshold`. Elle prepare une partie du script.

In [ ]:
def select_validation_threshold(sweep, policy):
    val = sweep[sweep["split"].eq("val")].copy()
    if policy == "balanced_f1":
        return val.sort_values(["window_f1", "danger_clip_hit_rate", "safe_false_alarms_per_min"], ascending=[False, False, True]).iloc[0]
    if policy == "max_hit_low_fa":
        return val.sort_values(["danger_clip_hit_rate", "safe_false_alarms_per_min", "window_precision"], ascending=[False, True, False]).iloc[0]
    if policy == "precision_guard":
        eligible = val[val["window_precision"] >= 0.50]
        if eligible.empty:
            eligible = val
        return eligible.sort_values(["danger_clip_hit_rate", "safe_false_alarms_per_min", "window_precision"], ascending=[False, True, False]).iloc[0]
    raise ValueError(policy)


## Fonction `apply_threshold`

Cette cellule definit `apply_threshold`. Elle prepare une partie du script.

In [ ]:
def apply_threshold(df, score_col, variant, threshold, policy, seed, persistence_windows):
    test = df[df["split"].eq("test")].copy()
    sweep = threshold_sweep(test.rename(columns={score_col: "risk"}), "risk", 1.0, "test", persistence_windows=persistence_windows)
    exact = sweep.iloc[(sweep["threshold"] - threshold).abs().argsort().iloc[0]]
    y = test["danger_within_1.0s"].astype(int).to_numpy()
    p = test[score_col].astype(float).to_numpy()
    return {
        "repeat_seed": seed,
        "variant": variant,
        "policy": policy,
        "selected_threshold": float(threshold),
        "test_average_precision": safe_auc(average_precision_score, y, p),
        "test_roc_auc": safe_auc(roc_auc_score, y, p),
        "test_brier": float(brier_score_loss(y, p)) if len(np.unique(y)) > 1 else np.nan,
        "test_ece_10bin": ece_score(y, p),
        "test_hit_rate": float(exact["danger_clip_hit_rate"]),
        "test_false_alarms_per_min": float(exact["safe_false_alarms_per_min"]),
        "test_window_precision": float(exact["window_precision"]),
        "test_window_recall": float(exact["window_recall"]),
        "test_window_f1": float(exact["window_f1"]),
        "test_median_early_warning_s": exact["median_early_warning_s"],
    }


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(df, group_cols, metric_cols):
    rows = []
    for keys, group in df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        row["n_repeats"] = int(group["repeat_seed"].nunique())
        for col in metric_cols:
            row[f"{col}_mean"] = float(pd.to_numeric(group[col], errors="coerce").mean())
            row[f"{col}_std"] = float(pd.to_numeric(group[col], errors="coerce").std(ddof=0))
        rows.append(row)
    return pd.DataFrame(rows)


## Fonction `write_summary`

Cette cellule definit `write_summary`. Elle prepare une partie du script.

In [ ]:
def write_summary(run_dir, metric_summary, operating_summary, config):
    lines = ["# Final Fusion Meta-Model Ablation", ""]
    lines.append("This audit compares final trajectory+attention+PPE fusion families using the existing same-parent-split final-score features. Meta-models train only on each seed's train split; thresholds are selected on validation and applied to test.")
    lines.append("")
    lines.append("## Model Families")
    lines.append("")
    for item in config["meta_models"]:
        lines.append(f"- `{item}`")
    lines.append("")
    lines.append("## Test AP Summary")
    lines.append("")
    lines.append("| rank | variant | AP mean | AP std | ROC AUC | Brier | ECE | hit | FA/min | precision | train-test AP gap |")
    lines.append("|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|")
    test = metric_summary[metric_summary["split"].eq("test")].copy()
    train = metric_summary[metric_summary["split"].eq("train")][["variant", "average_precision_mean"]].rename(columns={"average_precision_mean": "train_ap_mean"})
    test = test.merge(train, on="variant", how="left")
    test["train_test_gap"] = test["train_ap_mean"] - test["average_precision_mean"]
    for rank, (_, row) in enumerate(test.sort_values("average_precision_mean", ascending=False).head(20).iterrows(), start=1):
        lines.append(
            f"| {rank} | {row['variant']} | {row['average_precision_mean']:.3f} | {row['average_precision_std']:.3f} | "
            f"{row['roc_auc_mean']:.3f} | {row['brier_mean']:.3f} | {row['ece_10bin_mean']:.3f} | "
            f"{row['best_hit_rate_mean']:.3f} | {row['best_false_alarms_per_min_mean']:.3f} | "
            f"{row['best_window_precision_mean']:.3f} | {row['train_test_gap']:.3f} |"
        )
    lines.append("")
    lines.append("## Validation-Selected Operating Points")
    lines.append("")
    lines.append("| rank | variant | policy | threshold | AP | hit | FA/min | precision | F1 | median early s |")
    lines.append("|---:|---|---|---:|---:|---:|---:|---:|---:|---:|")
    view = operating_summary.copy()
    view["rank_score"] = (
        view["test_average_precision_mean"]
        + 0.8 * view["test_hit_rate_mean"]
        + 0.3 * view["test_window_precision_mean"]
        - 0.03 * view["test_false_alarms_per_min_mean"]
    )
    for rank, (_, row) in enumerate(view.sort_values("rank_score", ascending=False).head(24).iterrows(), start=1):
        lines.append(
            f"| {rank} | {row['variant']} | {row['policy']} | {row['selected_threshold_mean']:.2f} | "
            f"{row['test_average_precision_mean']:.3f} | {row['test_hit_rate_mean']:.3f} | "
            f"{row['test_false_alarms_per_min_mean']:.3f} | {row['test_window_precision_mean']:.3f} | "
            f"{row['test_window_f1_mean']:.3f} | {row['test_median_early_warning_s_mean']:.3f} |"
        )
    lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- This is a meta-fusion ablation over saved model scores, not a new pose/crop/sequence training run.")
    lines.append("- Nonlinear models can improve ranking on some splits, but train-test gaps are reported to expose overfitting.")
    lines.append("- The final score should remain a named operating policy. A higher AP meta-model is not automatically the safest deployment score if it worsens false alarms, precision, or early warning.")
    (run_dir / "final_fusion_meta_model_ablation_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    score_run = resolve(args.score_run)
    run_dir = make_run_dir(args.run_name)
    seeds = sorted({int(path.stem.replace("final_scores_seed", "")) for path in (score_run / "features").glob("final_scores_seed*.csv")})
    config = {
        "score_run": str(score_run),
        "seeds": seeds,
        "feature_sets": args.feature_sets,
        "meta_models": args.meta_models,
        "baseline_score_cols": BASELINE_SCORE_COLS,
        "persistence_windows": args.persistence_windows,
        "target": "danger_within_1.0s",
        "split_policy": "inherits same parent-video splits from final score run",
    }
    write_json(run_dir / "config.json", config)
    all_metrics = []
    all_sweeps = []
    all_selected = []
    for seed in seeds:
        df = pd.read_csv(score_run / "features" / f"final_scores_seed{seed}.csv")
        score_cols = []
        for col in BASELINE_SCORE_COLS:
            if col in df:
                score_cols.append(col)
        for feature_set in args.feature_sets:
            for model_name in args.meta_models:
                probs = fit_predict_meta(df, seed, feature_set, model_name, run_dir)
                if probs is None:
                    continue
                col = f"meta_{feature_set}_{model_name}"
                df[col] = probs
                score_cols.append(col)
        df.to_csv(run_dir / "features" / f"fusion_meta_scores_seed{seed}.csv", index=False)
        for col in score_cols:
            rows, sweeps = evaluate_score(df, col, col, seed, args.persistence_windows)
            all_metrics.extend(rows)
            sweep = pd.concat(sweeps, ignore_index=True)
            all_sweeps.append(sweep)
            for policy in ["balanced_f1", "max_hit_low_fa", "precision_guard"]:
                selected = select_validation_threshold(sweep, policy)
                all_selected.append(apply_threshold(df, col, col, float(selected["threshold"]), policy, seed, args.persistence_windows))
    metrics = pd.DataFrame(all_metrics)
    sweeps = pd.concat(all_sweeps, ignore_index=True) if all_sweeps else pd.DataFrame()
    selected = pd.DataFrame(all_selected)
    metrics.to_csv(run_dir / "metrics" / "fusion_meta_model_metrics.csv", index=False)
    sweeps.to_csv(run_dir / "metrics" / "fusion_meta_model_threshold_sweeps.csv", index=False)
    selected.to_csv(run_dir / "metrics" / "fusion_meta_model_validation_selected.csv", index=False)
    metric_summary = summarize(
        metrics,
        ["variant", "split"],
        ["average_precision", "roc_auc", "brier", "ece_10bin", "best_hit_rate", "best_false_alarms_per_min", "best_window_precision", "best_window_f1"],
    )
    operating_summary = summarize(
        selected,
        ["variant", "policy"],
        ["selected_threshold", "test_average_precision", "test_hit_rate", "test_false_alarms_per_min", "test_window_precision", "test_window_f1", "test_median_early_warning_s", "test_brier", "test_ece_10bin"],
    )
    metric_summary.to_csv(run_dir / "metrics" / "fusion_meta_model_metric_summary.csv", index=False)
    operating_summary.to_csv(run_dir / "metrics" / "fusion_meta_model_operating_summary.csv", index=False)
    write_summary(run_dir, metric_summary, operating_summary, config)
    append_report(run_dir, "Final Fusion Meta-Model Ablation", f"- Summary: `{run_dir / 'final_fusion_meta_model_ablation_summary.md'}`")
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Meta-model ablation for final trajectory+attention+PPE fusion scores.")
    parser.add_argument("--score-run", default="runs/exp_039_final_aggregated_score")
    parser.add_argument("--run-name", default="exp_068_final_fusion_meta_model_ablation")
    parser.add_argument("--feature-sets", nargs="+", default=["core", "all_scores"])
    parser.add_argument(
        "--meta-models",
        nargs="+",
        default=[
            "isotonic_sequence",
            "logistic_l2",
            "logistic_elasticnet",
            "gaussian_nb",
            "random_forest",
            "extra_trees",
            "hist_gbdt",
            "mlp_small",
            "xgb_conservative",
        ],
    )
    parser.add_argument("--persistence-windows", type=int, default=2)
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_068_final_fusion_meta_model_ablation_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["final_fusion_meta_model_ablation.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
